In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever

load_dotenv()

C:\Users\praja\AppData\Local\Temp\ipykernel_16632\3869711541.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


True

In [2]:
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [3]:
text = """
            LangChain is a framework for building LLM applications.

            RAG stands for Retrieval Augmented Generation.

            Vector databases store embeddings and enable semantic search.

            Transformers are the foundation of modern Large Language Models.
            """

In [5]:
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
documents=splitter.create_documents([text])

In [6]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db=FAISS.from_documents(documents,embeddings)

retriever=db.as_retriever(search_kwargs={"k":2})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

##### History Aware Retriever

In [7]:
contextualize_q_prompt=ChatPromptTemplate.from_messages([
    ("system",
     """
     Given a chat history and the latest user question,
     formulate a standalone question which can be understood
     without the chat history.
     """),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

In [8]:
history_aware_retriever=create_history_aware_retriever(llm,retriever,contextualize_q_prompt)

In [9]:
qa_prompt=ChatPromptTemplate.from_messages([
    ("system",
     """
     Answer the question based only on the provided context.

     Context:
     {context}
     """),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

In [10]:
document_chain=create_stuff_documents_chain(llm,qa_prompt)

In [11]:
conversational_rag=create_retrieval_chain(history_aware_retriever,document_chain)

In [12]:
chat_history=[]

In [13]:
response=conversational_rag.invoke({
    "input":"What is LangChain?",
    "chat_history":chat_history
})

print(response["answer"])

LangChain is a framework for building LLM (Large Language Model) applications.


In [14]:
chat_history.extend([
    ("human","What is LangChain?"),
    ("ai",response["answer"])
])

In [15]:
response=conversational_rag.invoke({
    "input":"What does it provide?",
    "chat_history":chat_history
})

print(response["answer"])

The context doesn't explicitly state what LangChain provides, only that it is a framework for building LLM applications.
